<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/cst2312_h11/blob/main/CST2312_PressRelease2Plot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Press Release to Plot

**Course:** CST 2312 Information and Data Management I
**Section:** H11, Fall 2026
**Pipeline:** SEC EDGAR full-text search, then Claude extraction, then geocoding, then a folium map

*A copy of this notebook is available at [bit.ly/cst2312_PressRelease2Plot](https://bit.ly/cst2312_PressRelease2Plot)*

## About This Notebook

Public companies announce store openings, store closings, and new facilities in press releases filed with the U.S. Securities and Exchange Commission (SEC). This notebook builds a five-stage pipeline that turns those press releases into points on a map:

| Stage | Task | Service | Session |
|:---|:---|:---|:---|
| 1 | Search recent 8-K filings for location keywords | SEC EDGAR full-text search | 23 |
| 2 | Download the press release text of each filing | SEC EDGAR archive | 23 |
| 3 | Ask Claude whether each filing announces a location event, and extract the details | Anthropic API | 23 |
| 4 | Convert each city and state into latitude and longitude | OpenStreetMap Nominatim | 24 |
| 5 | Plot the events on an interactive map | folium | 24 |

The notebook combines several skills from earlier modules: `requests` with query parameters and headers, JSON, Colab Secrets, the Claude API with a system prompt, and folium maps.

**Exercises.** Stages 1, 3, and 5 each contain one exercise, named S1, S3, and S5. Each exercise is broken into small steps. Every step has a code cell to complete and a check cell that confirms the result. A collapsible solution follows the last step of each exercise. Stages 2 and 4 are provided complete.

**Fallback files.** The pipeline depends on three outside services that can be slow or unavailable, and later stages depend on earlier ones. At the start of each stage, a cell checks whether the previous stage produced its result. If it did not, the cell loads a saved copy of that result from the course data repository and says so. A missing or failed step therefore does not prevent the rest of the notebook from running.

**Two sessions.** Session 23 covers Stages 1 to 3 and ends by saving the extracted events to a file. Session 24 starts in a new Colab session, reloads that file, and covers Stages 4 and 5. Run the notebook one cell at a time, from the top.

---
## Setup

The pipeline uses four libraries beyond the Python standard library:

- `requests` sends HTTP requests to SEC EDGAR and to OpenStreetMap.
- `anthropic` is the Python library for the Claude API.
- `folium` draws interactive maps.
- `beautifulsoup4` extracts plain text from the HTML of a press release.

Colab already includes `requests`. The cell below installs the other three.

In [ ]:
!pip install -q anthropic folium beautifulsoup4

### Colab Secrets

This notebook reads two values from Colab Secrets (the key icon in the left sidebar). Both must have **Notebook access** turned on.

| Secret name | Value |
|:---|:---|
| `ANTHROPIC_API_KEY` | Your Anthropic API key, as in Module 09 |
| `EMAIL_ADDRESS` | Your City Tech email address |

The email address identifies you to SEC EDGAR and to OpenStreetMap, as explained in the next section. Keeping it in Colab Secrets keeps it out of the notebook file.

In [ ]:
import os
from google.colab import userdata

# The Anthropic library reads the key from this environment variable
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# Contact address for the User-Agent header
EMAIL_ADDRESS = userdata.get("EMAIL_ADDRESS")

All of the imports used in this notebook:

In [ ]:
import json
import re
import time
from datetime import date, timedelta

import requests
import anthropic
import folium
from bs4 import BeautifulSoup

### The `User-Agent` Header

SEC EDGAR and OpenStreetMap Nominatim both require every request to carry a `User-Agent` header that names the program making the request and gives a way to contact the person responsible for it. A request without one is usually refused with the status code `403` (Forbidden).

The cell below builds a default `User-Agent` from your email address:

```
CST2312 Press Release to Plot (contact: your.name@citytech.cuny.edu)
```

To write your own instead, put it in `CUSTOM_USER_AGENT`. A valid `User-Agent` for these services has two parts:

1. A short name for the program or project, for example `Brooklyn Store Openings Map`.
2. A working email address at which you can be contacted.

When `CUSTOM_USER_AGENT` is left empty, the default is used.

In [ ]:
# Optional: write your own User-Agent text between the quotation marks.
# Include a project name and your email address.
CUSTOM_USER_AGENT = ""

DEFAULT_USER_AGENT = f"CST2312 Press Release to Plot (contact: {EMAIL_ADDRESS})"
USER_AGENT = CUSTOM_USER_AGENT.strip() or DEFAULT_USER_AGENT

REQUEST_HEADERS = {"User-Agent": USER_AGENT}
print("User-Agent:", USER_AGENT)

In [ ]:
# Check the User-Agent
assert EMAIL_ADDRESS and "@" in EMAIL_ADDRESS, \
    "EMAIL_ADDRESS is empty or is not an email address: check Colab Secrets"
assert "@" in USER_AGENT, "the User-Agent must contain an email address"
assert len(USER_AGENT) >= 15, "the User-Agent should name the project as well as give an email address"
print("OK: the User-Agent is ready.")

### Request Etiquette

SEC EDGAR and OpenStreetMap Nominatim are free public services. To remain available to everyone, each sets two rules:

1. **Identification.** Every request carries a `User-Agent` with contact information, so that the operator can reach the person responsible before blocking a misbehaving program.
2. **Rate limits.** SEC EDGAR accepts at most 10 requests per second. Nominatim accepts at most 1 request per second. The code in this notebook pauses with `time.sleep()` between requests to stay under both limits.

A program that breaks these rules is typically blocked for a period of time. Colab sessions share Google's network addresses, so a block can affect other students as well.

### Fallback Files Reference / Checksums

The following information can be used to confirm that the correct versions of the Fallback Files load.

| Measure | Value | FileName |
|:---|:---|:---|
| Run date | 2026-11-17 |
| Search window | 2026-10-18 to 2026-11-17 |
| Candidates found | 225 | <b>pr2p_candidates.json</b> |
| Filings downloaded | 50 | <b>pr2p_filings.json</b> |
| Events found | 5 | <b>pr2p_events.json</b> |
| Events geocoded | 4 | <b>pr2p_geocoded.json</b> |
| Input tokens | 163,293 | |
| Output tokens | 11,286 | |

 Use the literal file name from the `FileName` column above when running the following code to retrieve a Fallback File.  

### Fallback Files

The cell below defines `load_cached()`, which reads one of the saved result files from the course data repository on GitHub. The fallback cell at the start of each stage uses it.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/data/refs/heads/main/"


def load_cached(filename):
    """Download a saved pipeline result from the course data repository."""
    response = requests.get(DATA_URL + filename, timeout=60)
    response.raise_for_status()
    return response.json()

In [ ]:
# Example usage:
load_cached("pr2p_geocoded.json")

In [ ]:
def save_cached_file(filename):
    """Download a saved pipeline result and save it to the current working directory."""
    # Download the JSON data using your existing load_cached function
    data = load_cached(filename)

    # Save the JSON data to a local file with the same name
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2)

    print(f"Successfully saved {filename} to the current working directory.")


In [ ]:
# Example usage:
save_cached_file("pr2p_geocoded.json")

---
# Session 23: Retrieval and Extraction

---
## Stage 1: Search EDGAR for Candidate Filings

### Background: EDGAR and Form 8-K

EDGAR (Electronic Data Gathering, Analysis, and Retrieval) is the SEC's public filing system. Public companies must disclose material events to investors as they happen, and **Form 8-K** is the filing used for events between quarterly reports. Openings, closings, relocations, and expansions of significant facilities are among those events.

Each 8-K usually has attached documents called **exhibits**. A press release is usually attached as **Exhibit 99.1**.

EDGAR offers a free full-text search service at `efts.sec.gov/LATEST/search-index`. It requires no API key, only the `User-Agent` header.

Stage 1 asks EDGAR for every 8-K from the past 30 days that contains any of ten location phrases. Many of the results will not actually announce a location event. That is expected: Stage 1 aims to miss as few real events as possible, and Stage 3 removes the results that do not belong.

In [ ]:
# Each phrase is searched separately, and the results are combined afterward.
SEARCH_PHRASES = [
    '"new store"',
    '"new facility"',
    '"new location"',
    '"distribution center"',
    '"fulfillment center"',
    '"grand opening"',
    '"store closure"',
    '"store closing"',
    '"facility closure"',
    '"opens new"',
]

# A 30-day window ending today
WINDOW_DAYS = 30
WINDOW_END = date.today()
WINDOW_START = WINDOW_END - timedelta(days=WINDOW_DAYS)

EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"

print(f"Search window: {WINDOW_START} to {WINDOW_END} ({WINDOW_DAYS} days)")
print(f"Phrases to search: {len(SEARCH_PHRASES)}")

### Why Each Phrase Is Searched Separately

The SEC's documentation suggests that several phrases can be combined in one search with `OR`, for example `"new store" OR "new facility"`. In practice, combining many quoted phrases returns zero results, even when each phrase alone returns many. The dependable approach is to search each phrase separately and then combine the results.

A filing that matches two phrases appears in both result lists. Each search result carries a unique identifier in its `_id` field, so collecting the results in a dictionary keyed on `_id` keeps one copy of each filing.

The general lesson: documentation describes how a service is intended to behave, and testing shows how it actually behaves.

> **DISCUSSION PROMPT**
>
> Combining phrases with `OR` returned zero results, even though the documentation says it should work. How can a programmer confirm that an API behaves as its documentation describes before building a pipeline on it?

### Exercise S1: Build the EDGAR Search Function

The goal of this exercise is a function, `search_edgar_one_phrase()`, that searches EDGAR for one phrase and returns every result in the window. Steps 1 to 3 send a single request for one test phrase. Step 4 repeats that request across pages. Step 5 packages the work as a function.

**Pagination.** EDGAR returns at most 100 results per request, called a **page**. The `from` parameter tells EDGAR where to start: `from=0` returns results 1 to 100, `from=100` returns results 101 to 200, and so on. A program that needs more than 100 results sends one request per page and combines them. This is **pagination**.

#### Step 1: Build the parameter dictionary

Create a dictionary named `params` for the first page of results for `test_phrase`. It needs six keys:

| Key | Value |
|:---|:---|
| `q` | the search phrase |
| `dateRange` | `"custom"` |
| `startdt` | the window start as text, `WINDOW_START.isoformat()` |
| `enddt` | the window end as text, `WINDOW_END.isoformat()` |
| `forms` | `"8-K"` |
| `from` | the position of the first result: `page * 100` |

In [ ]:
test_phrase = '"new store"'
page = 0

# TODO: build the params dictionary with the six keys in the table above.
params = None

params

In [ ]:
# Step 1 check
assert isinstance(params, dict), "params must be a dictionary"
for key in ["q", "dateRange", "startdt", "enddt", "forms", "from"]:
    assert key in params, f"params is missing the key {key}"
assert params["q"] == test_phrase, "q should be test_phrase"
assert params["dateRange"] == "custom", 'dateRange should be "custom"'
assert params["startdt"] == WINDOW_START.isoformat(), "startdt should be WINDOW_START.isoformat()"
assert params["enddt"] == WINDOW_END.isoformat(), "enddt should be WINDOW_END.isoformat()"
assert params["forms"] == "8-K", 'forms should be "8-K"'
assert params["from"] == 0, "from should be page * 100, which is 0 for the first page"
print("OK: Step 1 passed.")

#### Step 2: Send one request

Send a GET request to `EDGAR_SEARCH_URL` with `params`, `REQUEST_HEADERS`, and `timeout=30`. Then call `response.raise_for_status()`, which raises an error if the status code reports a failure, so that a failed request stops the cell instead of passing along an empty result.

In [ ]:
# TODO: send the request and store it in response.
response = None

# TODO: call raise_for_status() on the response.


print("Status code:", response.status_code)

In [ ]:
# Step 2 check
assert response is not None, "replace None with the call to requests.get"
assert response.status_code == 200, f"EDGAR answered with status {response.status_code}"
assert "efts.sec.gov" in response.url, "the request should go to EDGAR_SEARCH_URL"
assert "from=0" in response.url, "the request should include the params dictionary"
print("OK: Step 2 passed.")

#### Step 3: Find the results in the JSON

The response is JSON. Two values are needed from it:

- The list of results for this page is at `data["hits"]["hits"]`.
- The total number of results in the window, across all pages, is at `data["hits"]["total"]["value"]`.

Store them in `hits` and `total`.

In [ ]:
data = response.json()

# TODO: store the list of results for this page in hits.
hits = None

# TODO: store the total number of results in the window in total.
total = None

print("Results on this page:", len(hits))
print("Results in the window:", total)

In [ ]:
# Step 3 check
assert isinstance(hits, list), "hits should be the list at data['hits']['hits']"
assert isinstance(total, int), "total should be the number at data['hits']['total']['value']"
assert len(hits) <= 100, "one page holds at most 100 results"
assert len(hits) <= total, "a page cannot hold more results than the window"
if hits:
    assert "_id" in hits[0] and "_source" in hits[0], "each result should have _id and _source keys"
print("OK: Step 3 passed.")

Each result is a dictionary. `_id` identifies the filing and the exhibit file, and `_source` holds details such as the company name and filing date:

In [ ]:
if hits:
    print(json.dumps(hits[0]["_id"], indent=2))
    print(json.dumps({key: hits[0]["_source"].get(key) for key in ["display_names", "file_date", "ciks"]}, indent=2))

#### Step 4: Repeat the request across pages

Write a loop over `page` from `0` up to `max_pages`. Inside the loop, repeat Steps 1 to 3 for that page, and add the page's results to `all_hits` with `extend`.

After adding the results, stop the loop once every result has been collected. Pages run from 0, so after page `page` the program holds at most `(page + 1) * 100` results. When that number is at least `total`, there are no more pages, and `break` ends the loop.

In [ ]:
max_pages = 2
all_hits = []
total = 0

for page in range(max_pages):
    # TODO: build params for this page (Step 1, with "from": page * 100).
    # TODO: send the request and call raise_for_status() (Step 2).
    # TODO: read the hits and the total from the JSON (Step 3).
    # TODO: add this page's hits to all_hits with all_hits.extend(...).
    # TODO: break out of the loop if (page + 1) * 100 >= total.
    pass

print("Results collected:", len(all_hits))
print("Results in the window:", total)

In [ ]:
# Step 4 check
assert total == data["hits"]["total"]["value"], "total was not updated inside the loop"
assert total == 0 or len(all_hits) > 0, "no results were added to all_hits"
assert len(all_hits) == min(total, max_pages * 100), \
    "all_hits should hold every result in the window, up to max_pages pages"
print("OK: Step 4 passed.")

#### Step 5: Package the work as a function

Complete `search_edgar_one_phrase()` so that it performs the Step 4 loop for any phrase and returns the pair `(all_hits, total)`.

Add one more line: `time.sleep(0.15)` after each page that is not the last, so that the function stays well under EDGAR's limit of 10 requests per second.

In [ ]:
def search_edgar_one_phrase(phrase, start_date, end_date, forms="8-K", max_pages=2):
    """Search EDGAR for one phrase and return (all_hits, total).

    EDGAR returns at most 100 results per page; at most max_pages pages are requested.
    """
    all_hits = []
    total = 0

    # TODO: copy the loop from Step 4, using phrase, start_date, end_date,
    #       and forms in place of test_phrase, WINDOW_START, WINDOW_END, and "8-K".
    # TODO: after the early-stop check, pause with time.sleep(0.15).
    raise NotImplementedError("Complete Exercise S1, Step 5.")

    return all_hits, total

In [ ]:
# Step 5 check
import inspect

source = inspect.getsource(search_edgar_one_phrase)
assert "raise_for_status" in source, "the function should call raise_for_status()"
assert "time.sleep" in source, "the function should pause with time.sleep(0.15) between pages"
assert ">=" in source, "the early stop should use >="

result = search_edgar_one_phrase(test_phrase, WINDOW_START, WINDOW_END)
assert isinstance(result, tuple) and len(result) == 2, "the function should return (all_hits, total)"
step5_hits, step5_total = result
assert len(step5_hits) == len(all_hits), "the function should collect the same results as Step 4"
print(f"OK: Step 5 passed. {len(step5_hits)} results for {test_phrase}.")

<details>
<summary><b>Solution: Exercise S1</b></summary>

```python
# Step 1
params = {
    "q": test_phrase,
    "dateRange": "custom",
    "startdt": WINDOW_START.isoformat(),
    "enddt": WINDOW_END.isoformat(),
    "forms": "8-K",
    "from": page * 100,
}

# Step 2
response = requests.get(EDGAR_SEARCH_URL, params=params, headers=REQUEST_HEADERS, timeout=30)
response.raise_for_status()

# Step 3
hits = data["hits"]["hits"]
total = data["hits"]["total"]["value"]

# Step 5 (Step 4 is the same loop without the function and the pause)
def search_edgar_one_phrase(phrase, start_date, end_date, forms="8-K", max_pages=2):
    """Search EDGAR for one phrase and return (all_hits, total)."""
    all_hits = []
    total = 0
    for page in range(max_pages):
        params = {
            "q": phrase,
            "dateRange": "custom",
            "startdt": start_date.isoformat(),
            "enddt": end_date.isoformat(),
            "forms": forms,
            "from": page * 100,
        }
        response = requests.get(EDGAR_SEARCH_URL, params=params, headers=REQUEST_HEADERS, timeout=30)
        response.raise_for_status()
        data = response.json()
        all_hits.extend(data["hits"]["hits"])
        total = data["hits"]["total"]["value"]
        if (page + 1) * 100 >= total:
            break
        time.sleep(0.15)
    return all_hits, total
```

The early stop compares `(page + 1) * 100` with `total`. After page 0, the program holds up to 100 results; if the window has 100 or fewer, there is no second page to request.

</details>

> **DISCUSSION PROMPT**
>
> The loop stops early when every result has been collected, and pauses between pages. What would happen to EDGAR, and to other students, if every student in the class ran a search loop with neither the early stop nor the pause?

### Search Every Phrase

The cell below applies the function to every phrase and keeps one copy of each filing, keyed on `_id`. If the search fails, for example because Exercise S1 is unfinished or EDGAR refused the requests, the cell reports the problem and the next cell loads the saved candidates.

In [ ]:
candidates = []

try:
    all_hits_by_id = {}
    print(f"{'Phrase':<25} {'Results in window':>18}")
    print("-" * 45)
    for phrase in SEARCH_PHRASES:
        phrase_hits, phrase_total = search_edgar_one_phrase(phrase, WINDOW_START, WINDOW_END)
        print(f"{phrase:<25} {phrase_total:>18}")
        for hit in phrase_hits:
            all_hits_by_id[hit["_id"]] = hit
        time.sleep(0.15)
    candidates = list(all_hits_by_id.values())
    print(f"\nUnique candidates after removing duplicates: {len(candidates)}")
except Exception as error:
    print("The live search did not complete:", type(error).__name__, error)

**Fallback.** If the cell above produced no candidates, the cell below loads the saved candidates.

In [ ]:
if candidates:
    print(f"Using {len(candidates)} candidates from the live search.")
else:
    candidates = load_cached("pr2p_candidates.json")
    print(f"Loaded {len(candidates)} saved candidates from the course data repository.")

### Inspect the Candidates

Before sending anything to Claude, look at what the search returned. The company names alone often suggest which filings are unlikely to concern a physical location.

In [ ]:
print(f"{'Date':<12} {'Company':<45} {'Exhibit file'}")
print("-" * 100)
for hit in candidates[:10]:
    source = hit["_source"]
    company = (source.get("display_names") or ["(unknown)"])[0]
    if len(company) > 43:
        company = company[:42] + "..."
    exhibit = hit["_id"].split(":")[-1]
    print(f"{source.get('file_date', ''):<12} {company:<45} {exhibit}")

---
## Stage 2: Download the Press Release Text

The search results describe each filing but do not contain the press release itself. Stage 2 downloads the exhibit file for each candidate and converts it to plain text for Claude. Stage 2 is provided complete.

### Where the exhibit file is

EDGAR stores every file at an address built from three pieces:

```
https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_dashes}/{filename}
```

- The **CIK** (Central Index Key) identifies the company. It is in the result's `_source["ciks"]`.
- The **accession number** identifies the filing. It is the part of `_id` before the colon, written without dashes in the address.
- The **filename** of the exhibit is the part of `_id` after the colon.

`build_exhibit_url()` assembles the address from those pieces.

### Plain text and truncation

Exhibits are HTML pages. HTML tags add length without adding meaning, so `fetch_exhibit_text()` uses the `BeautifulSoup` library to keep only the visible text. It then cuts any text longer than 8,000 characters. Most press releases are shorter than that and pass through unchanged. The limit keeps the cost of each Claude request predictable in Stage 3.

### How many filings

`MAX_FILINGS` sets how many candidates are downloaded and sent to Claude. A value of 50 keeps each run inexpensive and limits the number of requests the class sends to EDGAR.

In [ ]:
MAX_FILINGS = 50


def build_exhibit_url(hit):
    """Build the EDGAR archive address of the exhibit file for one search result."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_dashes}/{filename}"


def fetch_exhibit_text(hit, max_chars=8000):
    """Download an exhibit, reduce it to plain text, and shorten it to max_chars.

    Returns (text, url).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)

    if len(text) > max_chars:
        text = text[:max_chars] + " [truncated]"
    return text, url

Try the two functions on the first candidate:

In [ ]:
try:
    sample_text, sample_url = fetch_exhibit_text(candidates[0])
    print("Address:", sample_url)
    print("Length: ", len(sample_text), "characters")
    print("\nFirst 500 characters:\n" + sample_text[:500])
except Exception as error:
    print("The download did not complete:", type(error).__name__, error)

### Download the Exhibits

The loop below downloads up to `MAX_FILINGS` exhibits, pausing 0.15 seconds between requests. A failed download is reported and skipped.

In [ ]:
fetched_filings = []
fetch_failures = 0

for i, hit in enumerate(candidates[:MAX_FILINGS]):
    try:
        text, url = fetch_exhibit_text(hit)
        source = hit["_source"]
        fetched_filings.append({
            "company": (source.get("display_names") or ["(unknown)"])[0],
            "file_date": source.get("file_date"),
            "accession": hit["_id"].split(":")[0],
            "url": url,
            "text": text,
        })
    except Exception as error:
        fetch_failures += 1
        print(f"  Failed to download {hit['_id']}: {type(error).__name__}")
    time.sleep(0.15)  # stay under SEC's limit of 10 requests per second
    if (i + 1) % 25 == 0:
        print(f"Downloaded {i + 1} of {min(MAX_FILINGS, len(candidates))}")

print(f"\nDownloaded: {len(fetched_filings)}")
print(f"Failures:   {fetch_failures}")

**Fallback.** If no exhibits were downloaded, the cell below loads the saved ones.

In [ ]:
if fetched_filings:
    print(f"Using {len(fetched_filings)} filings downloaded in this session.")
else:
    fetched_filings = load_cached("pr2p_filings.json")[:MAX_FILINGS]
    print(f"Loaded {len(fetched_filings)} saved filings from the course data repository.")

---
## Stage 3: Classify and Extract with Claude

For each downloaded filing, the pipeline asks Claude two questions in one request:

1. Does this filing announce the opening, closing, relocation, or expansion of a specific physical facility?
2. If so, what kind of event is it, in which city and state, and what is a one-sentence summary?

Filings for which the answer to the first question is no are set aside. The rest go on to Stage 4.

### Structured output

Module 09 showed that Claude can be instructed to reply with a JSON object instead of prose. Later code can then read each field by name, for example `extraction["city"]`. Two practices make this dependable:

1. **An exact schema.** The system prompt names every field, its type, and, for fields with a fixed set of values, every allowed value.
2. **Defensive parsing.** Even with clear instructions, a reply occasionally arrives wrapped in markdown code fences or cannot be parsed. The extraction function removes fences and catches parsing errors, so that one bad reply does not stop the loop.

### Choosing the Model

Module 09 used Claude Opus, the most capable model family. Capability is not the only consideration. This stage performs a yes or no classification and extracts four short fields from a few thousand characters of text, once for each of 50 filings. That is a high-volume, well-defined task, which is what the smallest and least expensive family, Haiku, is designed for.

This notebook uses **Claude Haiku 5.5** (`claude-haiku-5-5`). The Anthropic documentation lists its base rates as $0.10 per million input tokens and $0.50 per million output tokens, a small fraction of the Opus rates. Rates change, so confirm current figures on the Anthropic pricing page before relying on them.

The general principle: match the model to the task. Reserve the most capable models for work that requires extended reasoning, and use smaller models for routine classification and extraction.

### Two details of Haiku 5.5

- **Thinking blocks.** Haiku 5.5 may reason before it answers. The reasoning arrives as one or more `thinking` blocks at the start of `response.content`, before the `text` block that holds the answer. Reading `response.content[0].text`, as in Module 09, can therefore fail. The `response_text()` helper below selects the block whose `type` is `"text"`.
- **Thinking counts toward `max_tokens`.** A small `max_tokens` value can be used up by thinking before any answer is written. The extraction call allows 2,000 tokens. It also omits `temperature`, which Haiku 5.5 does not accept.

In [ ]:
client = anthropic.Anthropic()
MODEL_ID = "claude-haiku-5-5"

# Running totals for estimating the cost of the run
TOKEN_USAGE = {"input": 0, "output": 0}


def response_text(response):
    """Return the text of the first text block in a Messages API response."""
    for block in response.content:
        if block.type == "text":
            return block.text
    return ""

### Exercise S3: Write the Extraction System Prompt

The system prompt decides whether Stage 3 succeeds. A vague prompt produces replies with inconsistent field names and values. A precise prompt produces records that the rest of the pipeline can rely on.

The prompt is built from four parts, one per step, and then combined in Step 5. Step 6 sends one filing to Claude with the finished prompt.

#### Step 1: The role statement

Write `ROLE_STATEMENT`: one or two sentences that tell Claude it is an analyst reviewing 8-K filing exhibits to identify announcements of location events. Name the four kinds of events (openings, closings, relocations, expansions) and give examples of physical facilities (stores, warehouses, distribution centers, offices, plants).

In [ ]:
# TODO: write the role statement.
ROLE_STATEMENT = ""

In [ ]:
# Step 1 check
text = ROLE_STATEMENT.lower()
assert "8-k" in text, "mention 8-K filings"
assert "analyst" in text, "give Claude the role of an analyst"
for word in ["opening", "closing", "relocation", "expansion"]:
    assert word in text, f"name the event kind: {word}"
print("OK: Step 1 passed.")

#### Step 2: The schema

Write `SCHEMA_SPECIFICATION`, which tells Claude to return a JSON object with exactly these five fields:

| Field | Type and allowed values |
|:---|:---|
| `is_location_event` | boolean. True only when the filing announces the opening, closing, relocation, or expansion of a specific facility at a named location. False for earnings, executive changes, financing, share repurchases, or general corporate updates. |
| `event_type` | one of `"opening"`, `"closing"`, `"relocation"`, `"expansion"`, `"other"`, or null |
| `city` | the city name, or null |
| `state` | the two-letter U.S. state code, such as `"NY"` or `"CA"`, or null if the location is outside the United States or not stated |
| `summary` | one sentence of fewer than 25 words describing the event, or null |

In [ ]:
# TODO: write the schema specification. A triple-quoted string allows several lines.
SCHEMA_SPECIFICATION = """"""

In [ ]:
# Step 2 check
for field in ["is_location_event", "event_type", "city", "state", "summary"]:
    assert field in SCHEMA_SPECIFICATION, f"the schema should name the field {field}"
for value in ["opening", "closing", "relocation", "expansion", "other"]:
    assert f'"{value}"' in SCHEMA_SPECIFICATION, f'list "{value}" as an allowed event_type, in double quotes'
assert "two-letter" in SCHEMA_SPECIFICATION.lower(), "require a two-letter state code"
assert "null" in SCHEMA_SPECIFICATION, "say which fields may be null"
print("OK: Step 2 passed.")

#### Step 3: The strictness clause

Write `STRICTNESS_CLAUSE`, which tells Claude to be strict: a location mentioned only in passing, such as the headquarters address in the standard closing paragraph of a press release, must produce `is_location_event: false`.

In [ ]:
# TODO: write the strictness clause.
STRICTNESS_CLAUSE = ""

In [ ]:
# Step 3 check
text = STRICTNESS_CLAUSE.lower()
assert "strict" in text, "tell Claude to be strict"
assert "passing" in text or "headquarters" in text, "describe locations mentioned only in passing"
assert "false" in text, "say that such filings return is_location_event: false"
print("OK: Step 3 passed.")

#### Step 4: The output clause

Write `OUTPUT_CLAUSE`, which tells Claude to return only the JSON object: no introductory sentence, no markdown code fences, and no explanation.

In [ ]:
# TODO: write the output clause.
OUTPUT_CLAUSE = ""

In [ ]:
# Step 4 check
text = OUTPUT_CLAUSE.lower()
assert "only" in text and "json" in text, "tell Claude to return only the JSON object"
assert "markdown" in text or "fence" in text, "rule out markdown code fences"
print("OK: Step 4 passed.")

#### Step 5: Combine the four parts

Join the four parts into `EXTRACTION_SYSTEM_PROMPT`, in order, with a blank line between parts. The string method `"\n\n".join([...])` joins a list of strings with a blank line between each pair.

In [ ]:
# TODO: join ROLE_STATEMENT, SCHEMA_SPECIFICATION, STRICTNESS_CLAUSE, and OUTPUT_CLAUSE.
EXTRACTION_SYSTEM_PROMPT = ""

print(EXTRACTION_SYSTEM_PROMPT)

In [ ]:
# Step 5 check
parts = [ROLE_STATEMENT, SCHEMA_SPECIFICATION, STRICTNESS_CLAUSE, OUTPUT_CLAUSE]
assert all(part.strip() for part in parts), "complete Steps 1 to 4 first: one of the parts is empty"
positions = [EXTRACTION_SYSTEM_PROMPT.find(part) for part in parts]
assert all(p >= 0 for p in positions), "the prompt should contain all four parts"
assert positions == sorted(positions), "the parts should appear in the order of Steps 1 to 4"
print("OK: Step 5 passed.")

#### Step 6: Send one filing to Claude

The function below sends one filing to Claude with `EXTRACTION_SYSTEM_PROMPT` and turns the reply into a dictionary. It is provided complete. Read how it handles the reply:

- `response_text()` selects the text block, skipping any thinking blocks.
- A `stop_reason` of `"refusal"` means the request was declined, so the function returns a non-event instead of failing.
- `re.sub()` removes markdown code fences if the model added them.
- `try` and `except` catch a reply that is not valid JSON.

Then complete the cell that calls the function on the first downloaded filing and stores the result in `sample_extraction`.

In [ ]:
def extract_with_claude(filing):
    """Send one filing to Claude and return the extracted fields as a dictionary."""
    user_message = f"Filing from {filing['company']} dated {filing['file_date']}:\n\n{filing['text']}"

    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=2000,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    )
    TOKEN_USAGE["input"] += response.usage.input_tokens
    TOKEN_USAGE["output"] += response.usage.output_tokens

    if response.stop_reason == "refusal":
        return {"is_location_event": False, "_error": "refusal"}

    raw = response_text(response).strip()
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"is_location_event": False, "_parse_error": raw[:200]}

In [ ]:
# TODO: call extract_with_claude() on the first downloaded filing.
sample_extraction = None

print(fetched_filings[0]["company"])
print(json.dumps(sample_extraction, indent=2))

In [ ]:
# Step 6 check
assert isinstance(sample_extraction, dict), "sample_extraction should be the dictionary returned by extract_with_claude()"
assert "_parse_error" not in sample_extraction, \
    "the reply was not valid JSON: review the output clause in Step 4"
assert isinstance(sample_extraction.get("is_location_event"), bool), "is_location_event should be true or false"
if sample_extraction["is_location_event"]:
    for field in ["event_type", "city", "state", "summary"]:
        assert field in sample_extraction, f"the reply is missing the field {field}"
    assert sample_extraction["event_type"] in ["opening", "closing", "relocation", "expansion", "other", None], \
        "event_type is not one of the allowed values: review the schema in Step 2"
print("OK: Step 6 passed.")

<details>
<summary><b>Solution: Exercise S3</b></summary>

```python
ROLE_STATEMENT = (
    "You are an analyst reviewing 8-K filing exhibits to identify announcements of "
    "location-related corporate events: openings, closings, relocations, or expansions "
    "of physical facilities (stores, warehouses, distribution centers, offices, plants)."
)

SCHEMA_SPECIFICATION = """Return ONLY a JSON object with these exact fields:
- is_location_event: boolean. True ONLY if the filing announces the opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, generic corporate updates, or mentions of locations that are not the subject of the announcement.
- event_type: one of "opening", "closing", "relocation", "expansion", "other", or null
- city: string with the city name, or null if no specific city is named
- state: two-letter US state code (for example "NY" or "CA"), or null if not US-based or not specified
- summary: one sentence (under 25 words) describing the event in plain language, or null"""

STRICTNESS_CLAUSE = (
    "Be strict. If the filing mentions a location only in passing (for example, the "
    "headquarters address in the boilerplate), return is_location_event: false."
)

OUTPUT_CLAUSE = "Return only the JSON object with no preamble, no markdown fences, and no explanation."

EXTRACTION_SYSTEM_PROMPT = "\n\n".join([ROLE_STATEMENT, SCHEMA_SPECIFICATION, STRICTNESS_CLAUSE, OUTPUT_CLAUSE])

sample_extraction = extract_with_claude(fetched_filings[0])
```

The schema lists the cases that must return false, rather than relying on a general instruction. Specific exclusions are followed more reliably than general ones.

</details>

> **DISCUSSION PROMPT**
>
> This stage uses Haiku 5.5, while Module 09 used Opus. Which characteristics of this task make the smaller model the appropriate choice? Describe a task in which Opus would be worth its higher cost.

### Recall and Precision

Two measures describe how well a search performs:

- **Recall** is the share of the relevant items that the search finds. If 20 filings in the window truly announce new stores and the search finds 18 of them, recall is 90 percent.
- **Precision** is the share of the items found that are relevant. If the search returns 100 candidates and 18 are real announcements, precision is 18 percent.

The two usually trade against each other. A narrower keyword list raises precision but misses some real events. A broader one finds more real events but admits more false matches.

This pipeline divides the work:

1. **Stage 1 aims for recall.** The broad keyword search finds nearly every relevant filing, along with many irrelevant ones.
2. **Stage 3 supplies the precision.** Claude reads each candidate and sets aside the ones that are not location events.

In earlier runs of this pipeline, roughly 12 to 18 percent of candidates were classified as location events. A low share is expected: it measures the filtering that Stage 3 performs.

### Run the Extraction on Every Filing

The loop below sends each downloaded filing to Claude, one request per filing, and keeps the filings classified as location events. Each kept record combines the filing details with the extracted fields.

In [ ]:
events = []
classification_counts = {"location_event": 0, "not_location_event": 0, "error": 0}

for i, filing in enumerate(fetched_filings[:MAX_FILINGS]):
    try:
        extraction = extract_with_claude(filing)
        if extraction.get("is_location_event"):
            classification_counts["location_event"] += 1
            events.append({**filing, **extraction})
        else:
            classification_counts["not_location_event"] += 1
    except Exception as error:
        classification_counts["error"] += 1
        print(f"  Extraction failed for {filing['company']}: {type(error).__name__}")
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1} of {min(MAX_FILINGS, len(fetched_filings))}")

print("\nClassification summary:")
for label, count in classification_counts.items():
    print(f"  {label}: {count}")
print(f"\nLocation events found: {len(events)}")
print(f"Tokens used: {TOKEN_USAGE['input']:,} input, {TOKEN_USAGE['output']:,} output")

**Fallback.** If the extraction found no events, for example because Exercise S3 is unfinished or the API key is missing, the cell below loads the saved events so that Session 24 can proceed.

In [ ]:
if events:
    print(f"Using {len(events)} events extracted in this session.")
else:
    events = load_cached("pr2p_events.json")
    print(f"Loaded {len(events)} saved events from the course data repository.")

### Inspect the Events

Read through the events before moving on. Check whether each one is a real location announcement and whether the city and state make sense. A pattern of wrong answers points to a part of the system prompt that needs revising.

In [ ]:
for event in events[:10]:
    print(f"{event['file_date']} | {event['company']}")
    print(f"  Type:    {event.get('event_type')}")
    print(f"  Where:   {event.get('city')}, {event.get('state')}")
    print(f"  Summary: {event.get('summary')}")
    print()

### Save the Events for Session 24

Session 24 begins in a new Colab session, in which the variables from today no longer exist. The cell below writes `events` to a file named `my_events.json` and downloads it to your computer. Keep the file for Session 24.

In [ ]:
from google.colab import files

with open("my_events.json", "w") as f:
    json.dump(events, f)

print(f"Saved {len(events)} events to my_events.json")
files.download("my_events.json")

---
# End of Session 23

---
# Session 24: Geocoding and Rendering

### Start of Session 24

1. Run every cell in the **Setup** section at the top of the notebook again: the installation, Colab Secrets, imports, `User-Agent`, and `load_cached()` cells.
2. Upload `my_events.json` from Session 23: open the **Files** panel (the folder icon in the left sidebar), select the upload icon, and choose the file.
3. Run the cell below. It loads your file if it is present, and the saved events otherwise.

In [ ]:
if os.path.exists("my_events.json"):
    with open("my_events.json") as f:
        events = json.load(f)
    print(f"Loaded {len(events)} events from my_events.json.")
else:
    events = load_cached("pr2p_events.json")
    print(f"my_events.json was not found. Loaded {len(events)} saved events from the course data repository.")

---
## Stage 4: Geocode the Locations

Each event has a city and state, but a map needs latitude and longitude. Converting a place description such as "Memphis, TN" into coordinates is called **geocoding**. Stage 4 is provided complete.

### OpenStreetMap Nominatim

Commercial geocoding services such as Google Maps require an API key and charge by volume. **Nominatim**, operated by the OpenStreetMap Foundation, is free and needs no key, but it requires a `User-Agent` and accepts at most one request per second. The loop below pauses 1.1 seconds between requests.

Two details of the code:

- **The state is part of the query.** Many city names occur in several states, so "Springfield" alone cannot be resolved. "Springfield, IL, USA" can.
- **Repeated places are looked up once.** The dictionary `coordinates_by_place` stores each result, so a city that appears in several events costs only one request.

Every query ends with `USA`, so a filing about a facility outside the United States usually fails to geocode and is left off the map.

In [ ]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"


def geocode_location(city, state):
    """Return (latitude, longitude) for a city and state, or None if not found."""
    if not city:
        return None
    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1}
    try:
        response = requests.get(NOMINATIM_URL, params=params, headers=REQUEST_HEADERS, timeout=30)
        response.raise_for_status()
        results = response.json()
        if results:
            return float(results[0]["lat"]), float(results[0]["lon"])
    except Exception as error:
        print(f"  Geocoding failed for '{query}': {type(error).__name__}")
    return None


geocoded_events = []
coordinates_by_place = {}

for event in events:
    place = (event.get("city"), event.get("state"))
    if place not in coordinates_by_place:
        coordinates_by_place[place] = geocode_location(*place)
        time.sleep(1.1)  # Nominatim accepts at most one request per second
    coordinates = coordinates_by_place[place]
    if coordinates:
        event["latitude"], event["longitude"] = coordinates
        geocoded_events.append(event)

print(f"Geocoded {len(geocoded_events)} of {len(events)} events")
print(f"Nominatim requests sent: {len(coordinates_by_place)}")

**Fallback.** If no events were geocoded, the cell below loads the saved geocoded events.

In [ ]:
if geocoded_events:
    print(f"Using {len(geocoded_events)} events geocoded in this session.")
else:
    geocoded_events = load_cached("pr2p_geocoded.json")
    print(f"Loaded {len(geocoded_events)} saved geocoded events from the course data repository.")

---
## Stage 5: Draw the Map

### Design choices

- **Base map: CartoDB Positron.** This muted gray base map lets colored markers stand out. The default OpenStreetMap tiles carry strong colors and many labels that compete with the markers.
- **Color by event type.** `EVENT_COLORS` assigns green to openings, red to closings, orange to relocations, blue to expansions, and gray to other events. Colors that match common associations are read more quickly.
- **Popups instead of labels.** With dozens of markers, labels that are always visible would overlap. A popup shows the details of one event when its marker is clicked, and a tooltip shows the company name when the pointer rests on the marker.
- **Centered on the data.** The map starts at the average latitude and longitude of the events.

The cell below sets up the colors, the center, and an empty map named `m`.

In [ ]:
EVENT_COLORS = {
    "opening": "green",
    "closing": "red",
    "relocation": "orange",
    "expansion": "blue",
    "other": "gray",
}

if geocoded_events:
    center_lat = sum(e["latitude"] for e in geocoded_events) / len(geocoded_events)
    center_lon = sum(e["longitude"] for e in geocoded_events) / len(geocoded_events)
else:
    center_lat, center_lon = 39.5, -98.35  # geographic center of the contiguous United States

m = folium.Map(location=[center_lat, center_lon], zoom_start=4, tiles="CartoDB positron")

sample_event = geocoded_events[0]
print("Map center:", round(center_lat, 3), round(center_lon, 3))
print("Sample event:", sample_event["company"], "|", sample_event.get("event_type"))

### Exercise S5: Build the Map Markers

Steps 1 to 4 build one marker for `sample_event`. Step 5 repeats the same work for every event.

#### Step 1: Look up the marker color

Look up the color for `sample_event` in `EVENT_COLORS`, using `sample_event.get("event_type")` as the key. Use `"gray"` when the event type is missing or not in the dictionary. The dictionary method `get(key, default)` returns `default` when `key` is not present.

In [ ]:
# TODO: look up the color, with "gray" as the default.
color = None

color

In [ ]:
# Step 1 check
expected = EVENT_COLORS.get(sample_event.get("event_type"), "gray")
assert color == expected, f"expected {expected!r} for event type {sample_event.get('event_type')!r}"
print("OK: Step 1 passed.")

#### Step 2: Write the popup HTML

Build a string named `popup_html` that shows, in this order:

1. the company name in bold (`<b>...</b>`), followed by a line break (`<br>`);
2. the filing date and the event type in italics (`<i>...</i>`), on one line, followed by `<br>`;
3. the summary, followed by `<br>`;
4. a link to the filing, `<a href='...' target='_blank'>View SEC filing</a>`, where `target='_blank'` opens the link in a new browser tab.

Place the whole popup inside `<div style='width:260px'>...</div>` so that it keeps a fixed width.

In [ ]:
# TODO: build the popup HTML for sample_event.
popup_html = ""

print(popup_html)

In [ ]:
# Step 2 check
assert popup_html.startswith("<div") and popup_html.endswith("</div>"), "wrap the popup in a <div>...</div>"
assert "width:260px" in popup_html.replace(" ", ""), "give the <div> a width of 260px"
assert f"<b>{sample_event['company']}</b>" in popup_html, "show the company name in bold"
assert "<i>" in popup_html and str(sample_event["file_date"]) in popup_html, "show the filing date in italics"
assert sample_event["url"] in popup_html, "link to the SEC filing"
assert "_blank" in popup_html, "open the link in a new tab with target='_blank'"
print("OK: Step 2 passed.")

#### Step 3: Build one marker

Create `marker` with `folium.Marker()`, using these arguments:

- `location`: `[sample_event["latitude"], sample_event["longitude"]]`
- `popup`: `folium.Popup(popup_html, max_width=300)`
- `tooltip`: the company name
- `icon`: `folium.Icon(color=color, icon="info-sign")`

In [ ]:
# TODO: create the marker.
marker = None

marker

In [ ]:
# Step 3 check
assert isinstance(marker, folium.Marker), "marker should be created with folium.Marker()"
assert list(marker.location) == [sample_event["latitude"], sample_event["longitude"]], \
    "the marker should be placed at the event's latitude and longitude"
print("OK: Step 3 passed.")

#### Step 4: Add the marker to the map

A marker does not appear on a map until it is added to one. Call `marker.add_to(m)`, then end the cell with `m` on its own line so that the map is displayed.

In [ ]:
# TODO: add the marker to the map.


m

In [ ]:
# Step 4 check
marker_count = sum(isinstance(child, folium.Marker) for child in m._children.values())
assert marker_count >= 1, "no marker has been added to m: call marker.add_to(m)"
print("OK: Step 4 passed.")

#### Step 5: Add a marker for every event

The cell below starts a new, empty map. Write a loop over `geocoded_events` that repeats Steps 1 to 4 for each `event`: look up the color, build the popup HTML, create the marker, and add it to `m`. Keep `m` as the last line of the cell.

In [ ]:
m = folium.Map(location=[center_lat, center_lon], zoom_start=4, tiles="CartoDB positron")

# TODO: loop over geocoded_events and add one marker per event.


m

In [ ]:
# Step 5 check
marker_count = sum(isinstance(child, folium.Marker) for child in m._children.values())
assert marker_count == len(geocoded_events), \
    f"the map has {marker_count} markers for {len(geocoded_events)} events: check that .add_to(m) is inside the loop"
print(f"OK: Step 5 passed. {marker_count} markers on the map.")

<details>
<summary><b>Solution: Exercise S5</b></summary>

```python
# Steps 1 to 4 for one event
color = EVENT_COLORS.get(sample_event.get("event_type"), "gray")
popup_html = (
    f"<div style='width:260px'>"
    f"<b>{sample_event['company']}</b><br>"
    f"<i>{sample_event['file_date']}, {sample_event.get('event_type')}</i><br>"
    f"{sample_event.get('summary')}<br>"
    f"<a href='{sample_event['url']}' target='_blank'>View SEC filing</a>"
    f"</div>"
)
marker = folium.Marker(
    location=[sample_event["latitude"], sample_event["longitude"]],
    popup=folium.Popup(popup_html, max_width=300),
    tooltip=sample_event["company"],
    icon=folium.Icon(color=color, icon="info-sign"),
)
marker.add_to(m)

# Step 5: the same work for every event
m = folium.Map(location=[center_lat, center_lon], zoom_start=4, tiles="CartoDB positron")
for event in geocoded_events:
    color = EVENT_COLORS.get(event.get("event_type"), "gray")
    popup_html = (
        f"<div style='width:260px'>"
        f"<b>{event['company']}</b><br>"
        f"<i>{event['file_date']}, {event.get('event_type')}</i><br>"
        f"{event.get('summary')}<br>"
        f"<a href='{event['url']}' target='_blank'>View SEC filing</a>"
        f"</div>"
    )
    folium.Marker(
        location=[event["latitude"], event["longitude"]],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=event["company"],
        icon=folium.Icon(color=color, icon="info-sign"),
    ).add_to(m)
m
```

</details>

> **DISCUSSION PROMPT**
>
> The marker colors come from a dictionary rather than a chain of `if` statements. What would each approach require if the pipeline grew from five event types to fifteen?

---
## Evaluation Notes

After a complete run, record these four observations:

1. **Candidates found.** `len(candidates)`, the number of filings the keyword search returned in the window.
2. **Share classified as events.** `len(events) / len(fetched_filings)`. A share below about 8 percent suggests that the keyword list is too broad. `"new location"` is the phrase to remove first.
3. **Geocoding success.** `len(geocoded_events) / len(events)`. A low share usually means Claude returned places that Nominatim cannot resolve: locations outside the United States, neighborhoods, or general regions.
4. **Cost.** The token totals printed by the extraction loop, multiplied by the current Haiku 5.5 rates, give an estimate. The Usage page of the Anthropic Console shows the actual charge.

In [ ]:
# Session 23 values exist only if this Colab session ran Session 23.
counts = {}
for name in ["candidates", "fetched_filings", "events", "geocoded_events"]:
    value = globals().get(name)
    counts[name] = len(value) if value is not None else None
    print(f"{name:<16} {counts[name] if value is not None else 'not in this Colab session'}")

if counts["fetched_filings"] and counts["events"] is not None:
    print(f"\nShare classified as events: {100 * counts['events'] / counts['fetched_filings']:.1f} percent")
if counts["events"]:
    print(f"Geocoding success:          {100 * counts['geocoded_events'] / counts['events']:.1f} percent")

---
## Glossary

| Term | Definition |
|:---|:---|
| **8-K** | The SEC form a public company files to report a material event between its quarterly reports. |
| **Accession number** | The unique identifier of one EDGAR filing. |
| **CIK** | Central Index Key: the SEC's unique identifier for a company. |
| **EDGAR** | Electronic Data Gathering, Analysis, and Retrieval: the SEC's public filing system. |
| **Exhibit** | A document attached to a filing. Press releases are usually Exhibit 99.1. |
| **Fallback file** | A saved copy of a pipeline result, loaded when a stage cannot produce its own result. |
| **Geocoding** | Converting a place description, such as a city and state, into latitude and longitude. |
| **Nominatim** | The free geocoding service operated by the OpenStreetMap Foundation. |
| **Pagination** | Retrieving a large result set in fixed-size pages, one request per page. |
| **Precision** | The share of the items a search returns that are relevant. |
| **Rate limit** | The maximum number of requests a service accepts in a period of time. |
| **Recall** | The share of all relevant items that a search returns. |
| **Structured output** | A model reply in a fixed format, such as a JSON object with named fields, that code can read directly. |
| **System prompt** | Standing instructions that govern how the model handles every message in a request. |
| **Thinking block** | A part of a Claude response that holds the model's reasoning, placed before the text block that holds its answer. |
| **Tooltip** | Text shown when the pointer rests on a map marker. |
| **User-Agent** | An HTTP header that names the program sending a request and, for these services, a contact address. |

---
## References

- SEC, [EDGAR full-text search](https://efts.sec.gov/LATEST/search-index) (the service used in Stage 1) and [Accessing EDGAR data](https://www.sec.gov/os/accessing-edgar-data) (the `User-Agent` and rate-limit requirements).
- OpenStreetMap Foundation, [Nominatim usage policy](https://operations.osmfoundation.org/policies/nominatim/).
- Anthropic, [Claude documentation](https://docs.claude.com) and [Claude Haiku 5.5 migration guide](https://platform.claude.com/docs/en/models/haiku-5-5/migration-guide) (thinking blocks, `max_tokens`, and sampling parameters).
- folium, [documentation](https://python-visualization.github.io/folium/).